In [79]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM

In [9]:
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
TOP_K = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
embeddings = np.load(ARTIFACTS_DIR / "embeddings.npy")

metadata = pd.read_parquet(ARTIFACTS_DIR / "metadata.parquet")

config = json.load(open(ARTIFACTS_DIR / "embedding_config.json", "r"))

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)
print("Embedding model:", config["embedding_model"])

Embeddings: (248218, 384)
Metadata: (248218, 9)
Embedding model: BAAI/bge-small-en-v1.5


In [13]:
embedding_model = SentenceTransformer(config["embedding_model"], device = DEVICE)
print("Embedding model loaded:", embedding_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2292.30it/s]


Embedding model loaded: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [80]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype = "auto", device_map = "auto")
llm_model.eval()

Loading weights: 100%|██████████| 434/434 [00:07<00:00, 57.39it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [81]:
def retrieve(query, top_k = 5):
    query_embedding = embedding_model.encode([query], convert_to_numpy = True, normalize_embeddings = True)
    scores = cosine_similarity(query_embedding, embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]
    results = metadata.iloc[top_k].copy()

    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

In [82]:
TEXT_COLUMN = "document"

if TEXT_COLUMN not in metadata.columns:
    raise ValueError(
        f"Expected '{TEXT_COLUMN}' column, "
        f"but found: {metadata.columns.tolist()}"
    )

In [83]:
def build_context(results):

    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"""
SOURCE {i + 1}
Similarity: {row["similarity"]:.4f}

{row[TEXT_COLUMN]}
"""
        )

    return "\n".join(context_parts)

In [84]:
SYSTEM_PROMPT = """
You are MIRx, a medical information retrieval assistant.

Your task is to answer questions using ONLY the information
provided in the retrieved context.

Rules:

1. Do not invent information.
2. Do not introduce medical facts that are absent from the context.
3. If the retrieved context is insufficient, say so clearly.
4. Keep the answer concise and well structured.
5. When possible, mention the relevant drug or medical entity.
6. Do not make diagnoses or provide personalized medical treatment.
7. Clearly distinguish information found in the retrieved context
   from general uncertainty.
"""

In [90]:
def generate_with_qwen(query, context):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Retrieved medical information:

{context}

User question:

{query}

Using only the retrieved medical information above,
answer the user's question.
"""
        }
    ]

    # Create model inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move inputs to the model device
    inputs = inputs.to(
        next(llm_model.parameters()).device
    )

    # Number of input tokens
    input_length = inputs["input_ids"].shape[-1]

    # Generate
    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[
        0,
        input_length:
    ]

    # Decode
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [86]:
def generate_response(query, top_k = 5):

    # -------------------------
    # 1. Retrieve
    # -------------------------

    results = retrieve(query, top_k = top_k)

    # -------------------------
    # 2. Build context
    # -------------------------

    context = build_context(results)

    # -------------------------
    # 3. Generate answer
    # -------------------------

    answer = generate_with_qwen(query, context)

    return answer, results

In [91]:
query = "What are the alternatives to paracetamol?"

answer, sources = generate_response(query, top_k = 5)

print("=" * 80)
print("MIRx RESPONSE")
print("=" * 80)

print(answer)

print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, row in sources.iterrows():

    print(
        f"\n[{i + 1}] "
        f"Similarity: {row['similarity']:.4f}"
    )

    print(row["document"])

MIRx RESPONSE
The alternatives to paracetamol (P-Aminophenol Derivative) mentioned in the retrieved information include:

- Tifmol Oral Suspension
- Moltrex Oral Suspension
- Kyomol Suspension
- Nettmol 250mg Oral Suspension
- Padicaf 250mg Oral Suspension
- Answell 400 mg/325 mg Tablet
- Bruace 400 mg/325 mg Tablet
- Rupar 400 mg/325 mg Tablet
- Brufamol Tablet
- Zupar 400mg/325mg Tablet
- Ladex-P Syrup
- Emfort P Syrup
- Mefnoc P Syrup
- Mefnix P Syrup
- Parafen Syrup
- Mefcad P Syrup
- Multigon Tablet
- Ibuwin 400 mg/500 mg Tablet
- Tolfen Tablet
- Ibuflam 400 mg/500 mg Tablet
- Arden Plus 400 mg/500 mg Tablet


RETRIEVED SOURCES

[1] Similarity: 0.6817
name: dr best paracetamol 250 oral suspension Chemical Class: P-Aminophenol Derivative Habit Forming: No Therapeutic Class: PAIN ANALGESICS Action Class: Analgesic & Antipyretic-PCM Substitutes: Tifmol Oral Suspension, Moltrex Oral Suspension, Kyomol Suspension, Nettmol 250mg Oral Suspension, Padicaf 250mg Oral Suspension Side Effect

In [27]:
print(metadata.columns.tolist())

['name', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 'Substitutes', 'Side Effects', 'Uses', 'document']


In [28]:
display(metadata.head())

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses,document
0,augmentin 625 duo tablet,None,No,ANTI INFECTIVES,None,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,name: augmentin 625 duo tablet Habit Forming: ...
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,name: azithral 500 tablet Chemical Class: Macr...
2,ascoril ls syrup,None,No,RESPIRATORY,None,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,name: ascoril ls syrup Habit Forming: No Thera...
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,name: allegra 120mg tablet Chemical Class: Dip...
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions,name: avil 25 tablet Chemical Class: Pyridines...


In [92]:
test_queries = [
    "What are the alternatives to paracetamol?",
    "What are the side effects of ibuprofen?",
    "Which drugs are used for pain relief?",
    "What are the alternatives to aspirin?",
    "Which medicines are associated with nausea?"
]

for query in test_queries:

    print("\n")
    print("#" * 100)
    print("QUERY:", query)

    answer, sources = generate_response(
        query,
        top_k=5
    )

    print("\nANSWER:")
    print(answer)



####################################################################################################
QUERY: What are the alternatives to paracetamol?

ANSWER:
The alternatives to paracetamol (P-Aminophenol Derivative) mentioned in the retrieved information include:

- Tifmol Oral Suspension
- Moltrex Oral Suspension
- Kyomol Suspension
- Nettmol 250mg Oral Suspension
- Padicaf 250mg Oral Suspension
- Answell 400 mg/325 mg Tablet
- Bruace 400 mg/325 mg Tablet
- Rupar 400 mg/325 mg Tablet
- Brufamol Tablet
- Zupar 400mg/325mg Tablet
- Ladex-P Syrup
- Emfort P Syrup
- Mefnoc P Syrup
- Mefnix P Syrup
- Parafen Syrup
- Mefcad P Syrup
- Multigon Tablet
- Ibuwin 400 mg/500 mg Tablet
- Tolfen Tablet
- Ibuflam 400 mg/500 mg Tablet
- Arden Plus 400 mg/500 mg Tablet


####################################################################################################
QUERY: What are the side effects of ibuprofen?

ANSWER:
The side effects of ibuprofen (mentioned in both SOURCE 2 and SOURCE 3) i